# Caso DT-009 — Concentracion de la renta pesquera
## 🟢 Nivel NOVATO — Ver la concentracion con un grafico y dos indices simples.

**Curso:** IA Aplicada a la Produccion Pesquera · UTN FRCh · PesquerosEnIA · 2026
**Autor:** Ariel Giamportone — Ingeniero Pesquero, Cientifico de datos e IA, docente-investigador UTN FRTDF, fundador Pesqueros en IA
**Apoya a:** DT-ALGP-2026-009 (dependencia tecnologica naval y concentracion de renta en merluza negra) y DT-ALGP-2026-012 (captura silenciosa de la renta pesquera)

---

> **Frase ancla:** *La concentracion no se declama; se calcula. Un solo indice vuelve visible lo que el discurso mantiene difuso.*

**Que hace este caso.** La CITC (Cuota Individual Transferible de Captura) es un instrumento; su *aplicacion* puede concentrar la renta en pocas manos sin que eso sea evidente a simple vista. Aca convertimos esa afirmacion de economia politica en algo medible y reproducible: tomamos las participaciones en la CMP de merluza negra y calculamos indices de concentracion estandar.

**Declaracion de conflicto de interes.** La UTN FRTDF es beneficiaria *prima facie* de la Ley provincial 1545/2024 y de la Ley 1614/2026. Este material se produce en el marco docente-investigador del autor y no constituye asesoramiento a ninguna de las empresas mencionadas.


### Taxonomia de evidencia (invariante de la serie DT-ALGP)

Todo valor que usamos lleva una etiqueta. **Nunca llenamos un hueco con un numero inventado.**

| Clase | Significado |
|---|---|
| `CONFIRMADO` | Verificado en fuente oficial primaria |
| `ESTIMADO` | Calculado a partir de supuestos declarados |
| `REFERENCIAL` | Tomado de fuente secundaria, sujeto a revision |
| `HUECO` | Dato faltante, se marca y no se completa |

> Las participaciones de las tres empresas son **CONFIRMADO**: provienen del Anexo IF-2024-00000277-CFP-CFP de la **Resolucion CFP 4/2024** (Acta CFP 15/2024, Boletin Oficial 10-sep-2024). **No hay operadores menores**: el resto de la CMP es Reserva de Administracion (18,80%) y Fondo de Reasignacion (0,40%), que **no** son empresas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paleta institucional de la serie DT-ALGP
AZUL, ROJO, VERDE, NARANJA = '#08519c', '#a50f15', '#005a32', '#cc6600'
plt.rcParams.update({'figure.figsize': (10, 5.5), 'font.size': 11})
np.random.seed(42)
print('✓ Librerias cargadas')

In [ ]:
# -- Estructura de la CMP de merluza negra -- Resolucion CFP 4/2024 -------------
# CONFIRMADO: Anexo IF-2024-00000277-CFP-CFP - Acta CFP 15/2024 - BO 10-sep-2024.
# Las 3 empresas = 80,80 % de la CMP = 100 % de la CITC asignada a empresas.
# El resto NO son empresas: Reserva de Administracion + Fondo de Reasignacion.
titulares = pd.DataFrame({
    'operador':  ['Estremar / San Arawa', 'Argenova', 'Pesantar',
                  'Reserva de Administracion', 'Fondo de Reasignacion'],
    'share_pct': [37.83, 23.02, 19.94, 18.80, 0.40],
    'tipo':      ['Empresa', 'Empresa', 'Empresa',
                  'Reserva del Estado', 'Reserva del Estado'],
    'evidencia': ['CONFIRMADO', 'CONFIRMADO', 'CONFIRMADO', 'CONFIRMADO', 'CONFIRMADO'],
})
# Suma = 99,99 %: el 0,01 % es redondeo del propio acto (art. 1 declara 80,80 % de CITC;
# la suma exacta por buque del Anexo es 80,7915 %). Ambos valores son correctos.
assert abs(titulares['share_pct'].sum() - 100.0) < 0.05, 'La suma se aparta demasiado de 100 %'
titulares['share'] = titulares['share_pct'] / 100.0
empresas = titulares[titulares['tipo'] == 'Empresa']
print('Filas:', len(titulares), '| Suma =', round(titulares.share_pct.sum(), 2), '%')
print(f'CITC (3 empresas): {empresas.share_pct.sum():.2f} % de la CMP  = 100 % de la cuota asignada a empresas')
titulares

In [ ]:
# ── Funciones de concentracion (reutilizables) ────────────────────────────────
def concentration_ratio(shares, k):
    '''CRk: suma de las k mayores participaciones (en %).'''
    return np.sort(shares)[::-1][:k].sum() * 100

def hhi(shares):
    '''Herfindahl-Hirschman Index sobre participaciones en % (0–10000).'''
    return float(np.sum((np.asarray(shares) * 100) ** 2))

def lorenz(shares):
    '''Devuelve (x, y) de la curva de Lorenz a partir de participaciones.'''
    s = np.sort(np.asarray(shares))
    cum = np.cumsum(s) / s.sum()
    x = np.linspace(0, 1, len(s) + 1)
    y = np.concatenate([[0], cum])
    return x, y

def gini(shares):
    '''Coeficiente de Gini (0 = igualdad total, 1 = concentracion total).'''
    s = np.sort(np.asarray(shares, dtype=float))
    n = len(s)
    idx = np.arange(1, n + 1)
    return float((np.sum((2 * idx - n - 1) * s)) / (n * s.sum()))

print('✓ Funciones listas')

In [ ]:
sh = titulares['share'].values

cr1 = concentration_ratio(sh, 1)
cr3 = concentration_ratio(sh, 3)
cr4 = concentration_ratio(sh, 4)
indice_hhi = hhi(sh)
indice_gini = gini(sh)

print(f'CR1 (lider)          : {cr1:5.2f} %')
print(f'CR3 (top 3 empresas) : {cr3:5.2f} %   <- CITC = 80,80 % de la CMP (CONFIRMADO, Res. CFP 4/2024)')
print(f'CR4 (top 4)          : {cr4:5.2f} %')
print(f'HHI (sobre la CMP)   : {indice_hhi:6.0f}')
print(f'Gini (sobre la CMP)  : {indice_gini:5.3f}')

def banda(v, cortes):
    etiquetas = ['no concentrado', 'moderada', 'alta']
    return etiquetas[sum(v >= c for c in cortes)]

print()
print(f'HHI segun guia 2010 (1500 / 2500): {banda(indice_hhi, [1500, 2500])}')
print(f'HHI segun guia 2023 (1000 / 1800): {banda(indice_hhi, [1000, 1800])}')
print('Nota: estos indices se calculan sobre la distribucion de la CMP (incluye la reserva')
print('estatal). Entre EMPRESAS, las 3 titulares se llevan el 100 % de la CITC.')

**Lectura rapida.** El numero mas intuitivo es el **CR3**: la suma de las tres participaciones mas grandes. Tres empresas concentran el **80,80 %** de la CMP -- y el **100 %** de la cuota efectivamente asignada a empresas (el resto es reserva del Estado). El **HHI** queda en zona **alta** bajo las dos guias (2010 y 2023). El **Gini** baja respecto de una version con muchos actores chicos, justamente porque *no hay* actores chicos: esa ausencia **es** la concentracion.

In [ ]:
fig, ax = plt.subplots()
colores = [ROJO if t == 'Empresa' else '0.6' for t in titulares['tipo']]
ax.bar(titulares['operador'], titulares['share_pct'], color=colores, edgecolor='white')
ax.axhline(0, color='0.3', lw=0.8)
ax.set_ylabel('Participacion en la CMP (%)')
ax.set_title('Distribucion de la CMP de merluza negra (Res. CFP 4/2024)')
ax.tick_params(axis='x', rotation=20)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=ROJO, label='Empresa (CITC) - CONFIRMADO'),
                   Patch(color='0.6', label='Reserva del Estado (RA + Fondo)')])
plt.tight_layout(); plt.show()

In [ ]:
x, y = lorenz(titulares['share'].values)
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.plot([0, 1], [0, 1], '--', color='0.5', label='Igualdad perfecta')
ax.plot(x, y, color=ROJO, lw=2.5, marker='o', label=f'Observado (Gini = {gini(titulares.share.values):.3f})')
ax.fill_between(x, y, x, color=ROJO, alpha=0.10)
ax.set_xlabel('Proporcion acumulada de titulares')
ax.set_ylabel('Proporcion acumulada de la cuota')
ax.set_title('Curva de Lorenz de la cuota de merluza negra')
ax.legend(); ax.set_aspect('equal'); plt.tight_layout(); plt.show()

---
## Sintesis

En una sola figura (Lorenz) y dos numeros (HHI, Gini) mostramos lo que un parrafo puede diluir: la renta de la merluza negra esta fuertemente concentrada. Ese es, en clave de datos, el corazon de la **captura silenciosa de la renta pesquera** (DT-012). El instrumento (CITC) no es el problema; su aplicacion concentradora, medida y trazable, si es discutible como politica publica.

**Transferible.** Los mismos indices (CR, HHI, Gini, Lorenz) sirven para cualquier cuota, permiso o subsidio del sector. La tecnica es general; el dato manda.

---
### Que NO afirma este notebook

1. Las participaciones de las tres empresas son **CONFIRMADO** (Resolucion CFP 4/2024, Anexo IF-2024-00000277-CFP-CFP, BO 10-sep-2024). Este notebook no agrega operadores inexistentes: el resto de la CMP es Reserva de Administracion (18,80 %) + Fondo de Reasignacion (0,40 %).
2. No afirma que la concentracion sea ilegal: la CITC es un instrumento legitimo. Lo que se mide es el *grado* de concentracion resultante de su aplicacion.
3. No afirma causalidad politica: cuantifica un estado de situacion, no prueba intencion.
4. No afirma nada sobre la sostenibilidad biologica del stock (ese eje se trata en DT-007).

**Fuente primaria:** Resolucion CFP 4/2024 - Asignacion de CITC de merluza negra 2025-2039, Anexo IF-2024-00000277-CFP-CFP, Acta CFP 15/2024, Boletin Oficial 10-sep-2024.